In [ ]:
# Reference: https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker/python

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision
from base64 import b64decode, b64encode
from IPython.display import display, Javascript, Image, clear_output
import time

model_path = 'pose_landmarker_heavy.task'

In [2]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
PoseLandmarkerResult = mp.tasks.vision.PoseLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode

# Global variable to store the latest MediaPipe result
latest_result = None

# Function to calculate the angle between three joints
def calculate_angle(a, b, c):
    """Calculates the angle between three points (a, b, c). 'b' is the vertex."""
    a = np.array([a.x, a.y]) # First point (Shoulder)
    b = np.array([b.x, b.y]) # Second point (Elbow)
    c = np.array([c.x, c.y]) # End point (Wrist)

    # Calculate radians using arctangent and convert to degrees
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    # Ensure the angle is never greater than 180 degrees
    if angle > 180.0:
        angle = 360.0 - angle

    return angle
    

In [3]:
# Store the result globally instead of printing it
def get_result(result: PoseLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
    global latest_result
    latest_result = result

In [4]:

# Create a pose landmarker instance with the live stream mode:
def print_result(result: PoseLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
    clear_output(wait=True)
    
    if result.pose_landmarks:
        # Get the first person detected
        landmarks = result.pose_landmarks[0]
        
        # Pull specific joints for a Bicep Curl or Lateral Raise
        # Landmark 11 = Left Shoulder, 13 = Left Elbow, 15 = Left Wrist
        l_shoulder = landmarks[11]
        r_shoulder = landmarks[12]

        l_elbow = landmarks[13]
        r_elbow = landmarks[14]
        
        l_wrist = landmarks[15]
        r_wrist = landmarks[16]


        print(f"--- Frame: {timestamp_ms}ms ---")
        print(f"Left Shoulder: ({l_shoulder.x:.2f}, {l_shoulder.y:.2f})")
        print(f"Right Shoulder: ({r_shoulder.x:.2f}, {r_shoulder.y:.2f})")
        
        print()
        
        print(f"Left Elbow:    ({l_elbow.x:.2f}, {l_elbow.y:.2f})")
        print(f"Right Elbow:    ({r_elbow.x:.2f}, {r_elbow.y:.2f})")
        
        print()
        
        print(f"Left Wrist:    ({l_wrist.x:.2f}, {l_wrist.y:.2f})")
        print(f"Right Wrist:    ({r_wrist.x:.2f}, {r_wrist.y:.2f})")



In [5]:
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=get_result,
    output_segmentation_masks=False)

cap = cv2.VideoCapture(0)

In [6]:
with PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Convert OpenCV BGR to MediaPipe RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Convert to MediaPipe Image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Generate monotonic timestamp in milliseconds
        frame_timestamp_ms = int(time.time() * 1000)

        # Send frame to the landmarker (Asynchronous)
        landmarker.detect_async(mp_image, frame_timestamp_ms)

        # DRAWING THE UI: Check if we have data from the callback
        if latest_result and latest_result.pose_landmarks:
            landmarks = latest_result.pose_landmarks[0]

            # Multiply the frame's width and height to get actual pixles on the screen
            h, w, _ = frame.shape

            # Get Right Arm Landmarks
            r_shoulder = landmarks[12]
            r_elbow = landmarks[14]
            r_wrist = landmarks[16]

            # Convert (X, Y) pixel coordinates
            shoulder_px = (int(r_shoulder.x * w), int(r_shoulder.y * h))
            elbow_px = (int(r_elbow.x * w), int(r_elbow.y * h))
            wrist_px = (int(r_wrist.x * w), int(r_wrist.y * h))

            # Calculate the elbow angle
            elbow_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)

            # Draw the lines (Bones)
            cv2.line(frame, shoulder_px, elbow_px, (255, 255, 255), 4) # White line, 4px thick
            cv2.line(frame, elbow_px, wrist_px, (255, 255, 255), 4)

            # Draw the dots (Joints)
            cv2.circle(frame, shoulder_px, 8, (0, 255, 0), -1) # Green dot, 8px radius
            cv2.circle(frame, elbow_px, 8, (0, 255, 0), -1)
            cv2.circle(frame, wrist_px, 8, (0, 255, 0), -1)

            # Display the angle text directly over the elbow
            cv2.putText(frame, str(int(elbow_angle)),
                        (elbow_px[0] + 20, elbow_px[1]), # Offset text slightly from the dot
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2, cv2.LINE_AA) # Cyan text

        # Display the raw webcam feed
        cv2.imshow('MediaPipe Prototype - Press Q to Quit', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()